# Welfare Scheme Leakage Detection

detect potential fund leakage in welfare scheme disbursements — situations where money
was released but didn't reach proportional beneficiaries (ghost/inflated claims).

data: `scheme_disbursements.csv` — 3600 rows, one per scheme-district-quarter

In [1]:
import csv
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from collections import defaultdict
from datetime import datetime
from pathlib import Path

DATA = Path('..') / 'data'

with open(DATA / 'scheme_disbursements.csv', newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print('rows:', len(rows))
print('cols:', list(rows[0].keys()))

rows: 3600
cols: ['id', 'scheme_id', 'district', 'fiscal_year', 'quarter', 'planned_amount_lakhs', 'actual_disbursed_lakhs', 'planned_beneficiaries', 'actual_beneficiaries', 'disbursement_date', 'mode']


In [2]:
# peek at the data
for r in rows[:4]:
    print(r)

{'id': '1', 'scheme_id': '1', 'district': 'Ahmednagar', 'fiscal_year': '2024-25', 'quarter': '1', 'planned_amount_lakhs': '502.998', 'actual_disbursed_lakhs': '126.67', 'planned_beneficiaries': '12500', 'actual_beneficiaries': '3147', 'disbursement_date': '2024-03-20', 'mode': 'DBT-Bank'}
{'id': '2', 'scheme_id': '1', 'district': 'Ahmednagar', 'fiscal_year': '2024-25', 'quarter': '2', 'planned_amount_lakhs': '502.998', 'actual_disbursed_lakhs': '135.61', 'planned_beneficiaries': '12500', 'actual_beneficiaries': '3370', 'disbursement_date': '2024-06-13', 'mode': 'DBT-Bank'}
{'id': '3', 'scheme_id': '1', 'district': 'Ahmednagar', 'fiscal_year': '2024-25', 'quarter': '3', 'planned_amount_lakhs': '502.998', 'actual_disbursed_lakhs': '134.04', 'planned_beneficiaries': '12500', 'actual_beneficiaries': '3331', 'disbursement_date': '2024-09-19', 'mode': 'DBT-Bank'}
{'id': '4', 'scheme_id': '1', 'district': 'Ahmednagar', 'fiscal_year': '2024-25', 'quarter': '4', 'planned_amount_lakhs': '502.998

In [3]:
# compute core ratios first — we need to see the distribution before picking thresholds
coverage_gaps  = []
cpb_ratios     = []
amount_covs    = []
bene_covs      = []

for r in rows:
    pa = float(r['planned_amount_lakhs'])
    aa = float(r['actual_disbursed_lakhs'])
    pb = float(r['planned_beneficiaries'])
    ab = float(r['actual_beneficiaries'])

    if pa <= 0 or pb <= 0:
        continue

    acr = min(aa / pa, 2.0)
    bcr = min(ab / pb, 2.0)
    gap = acr - bcr

    planned_cpb = pa / pb
    actual_cpb  = (aa / ab) if ab > 0 else (5.0 * planned_cpb if aa > 0 else planned_cpb)
    cpb_r = min(actual_cpb / max(planned_cpb, 1e-9), 10.0)

    coverage_gaps.append(gap)
    cpb_ratios.append(cpb_r)
    amount_covs.append(acr)
    bene_covs.append(bcr)

coverage_gaps = np.array(coverage_gaps)
cpb_ratios    = np.array(cpb_ratios)
amount_covs   = np.array(amount_covs)
bene_covs     = np.array(bene_covs)

def pct_summary(name, arr):
    print(f'{name:20s}  p5={np.percentile(arr,5):.3f}  p25={np.percentile(arr,25):.3f}  '
          f'p50={np.percentile(arr,50):.3f}  p75={np.percentile(arr,75):.3f}  '
          f'p95={np.percentile(arr,95):.3f}  max={arr.max():.3f}')

pct_summary('amount_coverage',   amount_covs)
pct_summary('bene_coverage',     bene_covs)
pct_summary('coverage_gap',      coverage_gaps)
pct_summary('cpb_ratio',         cpb_ratios)

amount_coverage       p5=0.038  p25=0.182  p50=0.241  p75=0.267  p95=0.297  max=0.872
bene_coverage         p5=0.037  p25=0.178  p50=0.240  p75=0.266  p95=0.296  max=0.872
coverage_gap          p5=0.000  p25=0.000  p50=0.000  p75=0.001  p95=0.003  max=0.006
cpb_ratio             p5=1.000  p25=1.000  p50=1.002  p75=1.006  p95=1.022  max=1.190


### Picking label thresholds based on the above distributions

leakage = `coverage_gap > p75(coverage_gap)` AND `amount_coverage > median`  
OR `cpb_ratio > p80(cpb_ratio)`  (paying notably more per person than expected)

this is data-driven so the positive rate stays sensible regardless of scale.

In [4]:
gap_thresh = np.percentile(coverage_gaps, 75)
amt_thresh = np.percentile(amount_covs, 50)    # above-median fund release
cpb_thresh = np.percentile(cpb_ratios, 80)

print(f'gap threshold (p75): {gap_thresh:.4f}')
print(f'amount_cov median  : {amt_thresh:.4f}')
print(f'cpb_ratio (p80)    : {cpb_thresh:.4f}')

# preview how many rows would be labelled leakage
ghost = np.sum((coverage_gaps > gap_thresh) & (amount_covs > amt_thresh))
inflated = np.sum(cpb_ratios > cpb_thresh)
total = len(coverage_gaps)
print(f'\nghost disbursement flag : {ghost} rows')
print(f'cost inflation flag     : {inflated} rows')
print(f'union (approx leakage)  : ~{ghost + inflated - int(ghost*inflated/total)} rows')

gap threshold (p75): 0.0008
amount_cov median  : 0.2410
cpb_ratio (p80)    : 1.0070

ghost disbursement flag : 426 rows
cost inflation flag     : 720 rows
union (approx leakage)  : ~1061 rows


## Feature engineering

In [5]:
# district risk score — historical avg coverage gap per district
dist_gaps = defaultdict(list)
for r in rows:
    pa = float(r['planned_amount_lakhs']); aa = float(r['actual_disbursed_lakhs'])
    pb = float(r['planned_beneficiaries']); ab = float(r['actual_beneficiaries'])
    if pa <= 0 or pb <= 0: continue
    dist_gaps[r['district']].append(min(aa/pa, 2.0) - min(ab/pb, 2.0))

dist_risk  = {d: float(np.mean(v)) for d, v in dist_gaps.items()}
max_dr     = max(abs(v) for v in dist_risk.values()) or 1.0

# scheme + district encoding
schemes   = sorted({int(r['scheme_id']) for r in rows})
s_to_idx  = {s: i for i, s in enumerate(schemes)}
n_schemes = len(schemes)

districts   = sorted({r['district'] for r in rows})
d_to_idx    = {d: i for i, d in enumerate(districts)}
n_districts = len(districts)

max_planned = max(float(r['planned_amount_lakhs']) for r in rows)

print(f'{n_schemes} schemes,  {n_districts} districts')

25 schemes,  36 districts


In [6]:
np.random.seed(42)
tf.random.set_seed(42)

X, y = [], []

for r in rows:
    pa = float(r['planned_amount_lakhs'])
    aa = float(r['actual_disbursed_lakhs'])
    pb = float(r['planned_beneficiaries'])
    ab = float(r['actual_beneficiaries'])

    if pa <= 0 or pb <= 0:
        continue

    acr  = min(aa / pa, 2.0)
    bcr  = min(ab / pb, 2.0)
    gap  = acr - bcr

    planned_cpb = pa / pb
    actual_cpb  = (aa / ab) if ab > 0 else (5.0 * planned_cpb if aa > 0 else planned_cpb)
    cpb_r = min(actual_cpb / max(planned_cpb, 1e-9), 10.0)

    is_dbt  = 1.0 if 'DBT' in r.get('mode', '') else 0.0
    quarter = int(r['quarter'])
    is_q4   = 1.0 if quarter == 4 else 0.0
    is_q1   = 1.0 if quarter == 1 else 0.0

    plan_norm  = pa / max_planned
    scheme_n   = s_to_idx.get(int(r['scheme_id']), 0) / max(n_schemes - 1, 1)
    district_n = d_to_idx.get(r['district'], 0) / max(n_districts - 1, 1)

    dr = dist_risk.get(r['district'], 0.0)
    dr_norm = (dr / max_dr + 1.0) / 2.0

    zero_bene = 1.0 if (ab == 0 and aa > 0) else 0.0
    over_disb = 1.0 if aa > pa * 1.05 else 0.0

    features = [
        acr / 2.0,
        bcr / 2.0,
        (gap + 2.0) / 4.0,
        cpb_r / 10.0,
        is_dbt,
        is_q4, is_q1,
        plan_norm,
        scheme_n,
        district_n,
        dr_norm,
        zero_bene,
        over_disb,
    ]

    # data-driven label — computed thresholds from above
    is_leakage = (
        (gap > gap_thresh and acr > amt_thresh) or
        (cpb_r > cpb_thresh)
    )

    X.append(features)
    y.append(1 if is_leakage else 0)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f'samples  : {len(y)}')
print(f'leakage  : {int(y.sum())} ({y.mean()*100:.1f}%)')

samples  : 3600
leakage  : 999 (27.8%)


## Model

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)

n_neg = (y_train == 0).sum(); n_pos = (y_train == 1).sum()
w = n_neg / max(n_pos, 1)
print(f'weight  1:{w:.2f}  (neg={n_neg}  pos={n_pos})')

weight  1:2.60  (neg=2081  pos=799)


In [8]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1, activation='sigmoid')
], name='leakage_detector')

model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

model.summary()

Model: "leakage_detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,929 (50.50 KB)

 Trainable params: 12,545 (49.00 KB)

 Non-trainable params: 384 (1.50 KB)

In [9]:
cbs = [
    keras.callbacks.EarlyStopping(monitor='val_auc', patience=15,
                                  restore_best_weights=True, mode='max'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                      patience=7, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=64,
    class_weight={0: 1.0, 1: w},
    callbacks=cbs,
    verbose=1
)

Epoch 1/150


 1/39 ━━━━━━━━━━━━━━━━━━━━ 1:09 2s/step - auc: 0.5306 - loss: 1.2300 - precision: 0.1500 - recall: 0.6000

31/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.5769 - loss: 1.1575 - precision: 0.2857 - recall: 0.7073 

39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - auc: 0.7020 - loss: 0.9851 - precision: 0.3528 - recall: 0.7974 - val_auc: 0.8955 - val_loss: 0.6084 - val_precision: 0.6331 - val_recall: 0.7458 - learning_rate: 3.0000e-04


Epoch 2/150


...

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - auc: 0.9912 - loss: 0.1554 - precision: 0.8698 - recall: 0.9809 - val_auc: 0.9898 - val_loss: 0.1275 - val_precision: 0.8358 - val_recall: 0.9492 - learning_rate: 7.5000e-05


Epoch 53/150


 1/39 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - auc: 0.9944 - loss: 0.2544 - precision: 0.5556 - recall: 1.0000

31/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.9906 - loss: 0.1759 - precision: 0.8409 - recall: 0.9599 

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.9917 - loss: 0.1593 - precision: 0.8710 - recall: 0.9618 - val_auc: 0.9897 - val_loss: 0.1281 - val_precision: 0.8358 - val_recall: 0.9492 - learning_rate: 7.5000e-05


In [10]:
y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob) if len(set(y_test.tolist())) > 1 else 0.0
cm   = confusion_matrix(y_test, y_pred)

print(f'Accuracy  {acc:.4f}')
print(f'Precision {prec:.4f}')
print(f'Recall    {rec:.4f}')
print(f'F1        {f1:.4f}')
print(f'AUC-ROC   {auc:.4f}')
print(f'CM:\n{cm}')

Accuracy  0.9472
Precision 0.8584
Recall    0.9700
F1        0.9108
AUC-ROC   0.9864
CM:
[[488  32]
 [  6 194]]


In [11]:
model.save('leakage_detector.h5')

stats = {
    'model': 'leakage_detector',
    'framework': f'TensorFlow {tf.__version__}',
    'trained_at': datetime.utcnow().isoformat() + 'Z',
    'unit': 'scheme x district x quarter',
    'samples': int(len(y)),
    'leakage_rate_pct': round(float(y.mean()) * 100, 2),
    'label_thresholds': {
        'coverage_gap_p75': round(float(gap_thresh), 4),
        'amount_cov_median': round(float(amt_thresh), 4),
        'cpb_ratio_p80': round(float(cpb_thresh), 4),
    },
    'label_def': 'leakage = (gap > gap_p75 AND amount_cov > median) OR cpb_ratio > p80',
    'features': [
        'amount_coverage_ratio', 'bene_coverage_ratio', 'coverage_gap',
        'cpb_ratio', 'is_dbt', 'is_q4', 'is_q1',
        'planned_amount_norm', 'scheme_norm', 'district_norm',
        'district_risk_norm', 'zero_beneficiary_flag', 'over_disbursed_flag'
    ],
    'scaler_mean':  sc.mean_.tolist(),
    'scaler_scale': sc.scale_.tolist(),
    'performance': {
        'accuracy':  round(acc,  4),
        'precision': round(prec, 4),
        'recall':    round(rec,  4),
        'f1':        round(f1,   4),
        'auc_roc':   round(auc,  4),
    },
    'confusion_matrix': cm.tolist()
}

with open('leakage_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('saved  leakage_detector.h5  +  leakage_stats.json')

saved  leakage_detector.h5  +  leakage_stats.json
